In [7]:
import time
from pathlib import Path

import pandas as pd
import requests
from google.cloud import bigquery

PROJECT_ID = "radar-economia-br"
DATASET = "economia_br"
LOCATION = "US"

SERIES = {
    "ipca_mensal": 433,
    "selic_meta": 432,
    "dolar_venda": 1,
    "desemprego": 24369,
    "ibc_br": 24363,
}

DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

In [8]:
def baixar_serie(codigo, ano_inicio=2010, ano_fim=None, bloco=5):
    ano_fim = ano_fim or pd.Timestamp.today().year
    partes = []

    for ano in range(ano_inicio, ano_fim + 1, bloco):
        ini = f"01/01/{ano}"
        fim = f"31/12/{min(ano + bloco - 1, ano_fim)}"
        url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"

        for tentativa in range(3):
            try:
                r = requests.get(
                    url,
                    params={"formato": "json", "dataInicial": ini, "dataFinal": fim},
                    timeout=60,
                )
                r.raise_for_status()
                dados = r.json()
                if dados:
                    partes.append(pd.DataFrame(dados))
                break
            except requests.RequestException as e:
                print(f"  tentativa {tentativa + 1} falhou ({codigo}, {ini}-{fim}): {e}")
                time.sleep(2)

    if not partes:
        raise RuntimeError(f"Nenhum dado retornado para a série {codigo}")

    df = pd.concat(partes, ignore_index=True)
    df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
    df["valor"] = pd.to_numeric(df["valor"], errors="coerce")
    return df.drop_duplicates("data").sort_values("data").reset_index(drop=True)

In [9]:
dados = {}
for nome, codigo in SERIES.items():
    print(f"Baixando {nome} (série {codigo})...")
    dados[nome] = baixar_serie(codigo)
    df = dados[nome]
    print(f"  {len(df)} linhas | {df['data'].min().date()} até {df['data'].max().date()}")
    df.to_csv(DATA_DIR / f"{nome}.csv", index=False)

Baixando ipca_mensal (série 433)...
  201 linhas | 2010-01-01 até 2026-09-01
Baixando selic_meta (série 432)...
  6152 linhas | 2010-01-01 até 2026-11-04
Baixando dolar_venda (série 1)...
  4218 linhas | 2010-01-04 até 2026-10-19
Baixando desemprego (série 24369)...
  174 linhas | 2012-03-01 até 2026-08-01
Baixando ibc_br (série 24363)...
  200 linhas | 2010-01-01 até 2026-08-01


In [10]:
for nome, df in dados.items():
    print(nome, "| nulos:", df["valor"].isna().sum())

ipca_mensal | nulos: 0
selic_meta | nulos: 0
dolar_venda | nulos: 0
desemprego | nulos: 0
ibc_br | nulos: 0


In [11]:
for nome, df in dados.items():
    tabela = f"{PROJECT_ID}.{DATASET}.raw_{nome}"
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
    job = client.load_table_from_dataframe(df, tabela, job_config=job_config)
    job.result()
    print(f"{tabela}: {client.get_table(tabela).num_rows} linhas")

/Users/vidigal/Documents/radar-economia-br/.venv/lib/python3.14/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


radar-economia-br.economia_br.raw_ipca_mensal: 201 linhas


/Users/vidigal/Documents/radar-economia-br/.venv/lib/python3.14/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


radar-economia-br.economia_br.raw_selic_meta: 6152 linhas


/Users/vidigal/Documents/radar-economia-br/.venv/lib/python3.14/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


radar-economia-br.economia_br.raw_dolar_venda: 4218 linhas


/Users/vidigal/Documents/radar-economia-br/.venv/lib/python3.14/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


radar-economia-br.economia_br.raw_desemprego: 174 linhas


/Users/vidigal/Documents/radar-economia-br/.venv/lib/python3.14/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


radar-economia-br.economia_br.raw_ibc_br: 200 linhas
